In [1]:
%load_ext autoreload
%autoreload

In [1]:
import random

from mesh import *
from utils import *

Fontconfig warning: ignoring UTF-8: not a valid region tag
Matplotlib is building the font cache; this may take a moment.


# Mesh Management

The objective of this TP is to manipulate the data structure of a 3D mesh.
The implemented functions can be checked thanks to the different display functions.
These functions will be used in the following TP to calculate the main curvatures as well as the mean curvature and the Gauss curvature.

When working with 3D meshes, it is natural to represent the mesh structure using a dictionary that contains basic elements like faces, edges, vertices, and normals as lists:


```pyhton
    ## Storage
    dict_attribut_base = {}
    dict_attribut_base["faces"] = faces
    dict_attribut_base["edges"] = edges
    dict_attribut_base["vertices"] = vertices
```

However, while this representation is straightforward, it is not the most efficient when we need to perform operations involving vertex neighborhoods or edge traversal.
To effectively access and manipulate relationships between vertices and their neighbors, we need a more sophisticated data structure.
This is where the **half-edge** data structure [1] comes into play.


This data structure is defined by the following points (cf the figure and the code below):

* An edge is broken down into two oriented half-edges
* Each half-edge stores a pointer is stored to the opposite half-edge, the vertex to which it points, the face to which it belongs, and the next half-edge
* Each vertex stores a pointer to an outgoing half-edge and each face, a pointer to one of its half-edges.

This representation is a more convenient way to browse a mesh.

Here is the implementation that we are going to use in this lab (see beginning of `mesh.py`).

```python
class Vertex:
    def __init__(self, x, y, z):
        self.x = x
        self.y = y
        self.z = z
        self.outgoing_half_edge = None  # A reference to one of the outgoing half-edges

class HalfEdge:
    def __init__(self):
        self.vertex = None  # The vertex the half-edge points to
        self.opposite = None    # The twin half-edge going in the opposite direction
        self.next = None    # The next half-edge around the face
        self.face = None    # The face the half-edge borders

class Face:
    def __init__(self):
        self.halfedge = None  # A reference to one of the half-edges bordering the face
        self.normal = np.zeros((3,1))

class Mesh:
    def __init__(self, num_vertices, num_faces):
        self.vertices = np.empty(num_vertices, dtype=Vertex)  # Liste de tous les sommets du maillage
        self.half_edges = np.empty(3 * num_faces, dtype=HalfEdge)  # Liste de toutes les demi-arêtes du maillage
        self.faces = np.empty(num_faces, dtype=Face)  # Liste de toutes les faces du maillage
```


<center> '<img src="images/half-edge-data-structure.png" alt="Image">'</center>
<caption><center> Halfedges structure. </center></caption>


[1]: Muller, D. E.; Preparata, F. P. (1978). "Finding the Intersection of Two Convex Polyhedra". Theoretical Computer Science. 7 (2): 217–236. doi:10.1016/0304-3975(78)90051-8

In [2]:
# path to the folder where the meshes are stored
absolute_path = "./"

path_models3D = absolute_path + "models3D/"

## Change the name of the mesh you want to use
## in the following line
name_mesh = "icosphere"


Load a `file.obj` that will be stored as a mesh.

In [3]:
filename = path_models3D + name_mesh + ".obj"
mesh = create_half_edge_mesh(filename)  ## This mesh would be used until the end of the TP

## If you have a close surface, there must be the message: "number of halfedge error 0"

number of halfedge 3840 number of halfedge error 0


Check that the mesh error number is 0. If this is not the case, the mesh file may be the cause (faces badly oriented, surface not closed, ...)

The function display_wireframe allows to visualize this mesh

In [4]:
fig = display_wireframe(filename)
fig.show()

We will use interesting properties of half-edge mesh: the calculation can be expensive to search for a neighborhood.

# Normals

Implement the `get_halfedges()` function in `mesh.py`to get half edges of a face.

It can be tested with the code of the next cell.

In [5]:
# 3D cube 
fig = display_wireframe("models3D/cube4.obj")
mesh_cube = create_half_edge_mesh("models3D/cube4.obj")

list_he = get_halfedges(mesh_cube.faces[0])
# Display the halfedges
start_point_list = [[he.opposite.vertex.x,he.opposite.vertex.y,he.opposite.vertex.z] for he in list_he]
end_point_list = [[he.vertex.x,he.vertex.y,he.vertex.z] for he in list_he]
fig2 = print_half_edge(start_point_list, end_point_list)
fig.add_traces(fig2.data)
fig.show()

number of halfedge 36 number of halfedge error 0


Code a function `neighbors_faces()` to get the neighboring faces of each vertex.

In [6]:
# To clean the last figure
fig = display_wireframe(filename)
mesh = create_half_edge_mesh(filename)

# Random vertex
vertex = np.random.choice(mesh.vertices)
faces = neighbors_faces(vertex)
# Display the adjacent faces
list_he = []
for f in faces:
    list_he.extend(get_halfedges(f))
    
fig2 = display_neighbors_faces(vertex, list_he)
fig2.add_traces(fig.data)
fig2.show()

number of halfedge 3840 number of halfedge error 0


Now, we will calculate the normal at each vertex as the average of the normals of the adjacent faces.
Implement the function `compute_normal()`

In [8]:
# Coordinates of vertices
x=np.array([i.x for i in mesh.vertices])
y=np.array([i.y for i in mesh.vertices])
z=np.array([i.z for i in mesh.vertices])

# vectors of normals
normals = [compute_normal(i) for i in mesh.vertices]

# Displays
normal_slider(x, y, z, normals, scale=1)

interactive(children=(IntSlider(value=1, description='Percent', min=1), Output()), _dom_classes=('widget-inter…

FigureWidget({
    'data': [], 'layout': {'template': '...'}
})

# Direct neighbors

Find the direct neighbors of a vertex wtih the Breadth-First Search (BFS) method.

This method is a graph traversal algorithm used to explore nodes level by level. It starts from a given node (root) and explores all its direct neighbors before moving on to their neighbors, continuing in a breadth-first manner.

<center> '<img src="images/voisins.png" alt="Image" width = 100 height = 100>'</center>
<caption><center> Algortithm to find direct neighbors. </center></caption>

The function essentially performs a layered traversal:

1. It starts from a given vertex.

2. It explores all vertices at depth 1 (direct neighbors).

3. Then, it moves to depth 2 (neighbors of neighbors), and so on, until it reaches the required depth (rank).

4. It collects only the vertices at the specified depth and returns them.

Implement the function `find_direct_neighbors()` to get the list of direct neighbors for a given vertex.

Then implement the function `find_neighbors_of_rank()` to get a list of vertices that are exactly `rank` edges away from the given vertex.

In [7]:
neighbors_slider(x, y, z, mesh.vertices[0], filename, scale=2)

NameError: name 'x' is not defined

## Optional : Shortest path

With this half-edges structure, you can also find the shortest path between two peaks.

Two kind of weighting : 
1. the weight of an edge is 1
2. the weight of an edge is the distance between the vertices it connects.

Compute `dijksta` function to find the minimal path between two vertices.

In [ ]:
filename = "models3D/dinosaur.obj"
mesh = create_half_edge_mesh(filename)

start_node = mesh.vertices[10000]
end_node = mesh.vertices[12500]

In [ ]:
def dijkstra(mesh, start, w = "one"):
    if w == "one":
        weight = 1
    # Initialization of previous distances and nodes
    distances = {node: float('infinity') for node in mesh.vertices}
    distances[start] = 0
    previous_nodes = {node: None for node in mesh.vertices}
    priority_queue = [(0, start)]
    
    while priority_queue:
        current_distance, current_node = heapq.heappop(priority_queue)
        
        if current_distance > distances[current_node]:
            continue
        
        # Update the distances of neighboring nodes
        for neighbor in get_direct_neighbors(current_node):
            if w == "dist":
                weight = get_distance(current_node,neighbor)
            distance = current_distance + weight
            
            if distance < distances[neighbor]:
                distances[neighbor] = distance
                previous_nodes[neighbor] = current_node
                heapq.heappush(priority_queue, (distance, neighbor))
    
    return distances, previous_nodes


In [ ]:
# The weight of the edges is 1
path, distance = shortest_path(mesh, start_node, end_node, w="one")
path_indice = [i.indice for i in path]
print(f"The shortest path from {start_node.indice} to {end_node.indice} is: {path_indice} with a distance of {distance} and passing through {len(path_indice)} vertices")

fig = display_wireframe(filename)

fig2 = show_path(path)
fig2.add_traces(fig.data)
fig2.show()